<a href="https://colab.research.google.com/github/Prudhvilakshman1112/GEN-AI/blob/main/EXP_5_Fine_tune_GPT_2_or_GPT_Neo_on_a_custom_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cell 1: Setup and Dataset Loading
In this cell, we import the Hugging Face Trainer API and the datasets library. We load a "tiny" slice of the Amazon Polarity dataset (200 examples). This is ideal for a lab environment or a quick proof-of-concept to ensure the pipeline works without needing hours of training time.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (GPT2LMHeadModel, GPT2Tokenizer, Trainer,
                         TrainingArguments, DataCollatorForLanguageModeling)

# Configuration and Data Loading
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load a small sample of 200 reviews
dataset = load_dataset("amazon_polarity", split="train[:200]")
print(f"Dataset loaded with {len(dataset)} examples")

#Cell 2: Tokenization and Data Preprocessing
Before the model can learn, we must convert raw text into a format it understands. We combine the review "title" and "content" into a single string. We then use the GPT-2 tokenizer to truncate or pad these strings to a uniform length of 256 tokens. Finally, we split the data into Training (90%) and Test (10%) sets.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# Set padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def tokenize_function(examples):
    # Combine title and content for the model to learn context
    texts = [t + " " + c for t, c in zip(examples['title'], examples['content'])]
    return tokenizer(texts, truncation=True, max_length=256, padding="max_length")

# Map the function over the dataset and split
tokenized = dataset.map(tokenize_function, batched=True,
                        remove_columns=dataset.column_names).train_test_split(0.1)

# Data Collator handles batching for Language Modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("Tokenization and splitting completed.")

#Cell 3: Configuring the Trainer and Fine-Tuning
This cell defines the TrainingArguments, which specify the "rules" of the training (learning rate, number of epochs, and batch size). We then initialize the Trainer—a high-level feature that handles the complex training loop for us. Running trainer.train() starts the actual learning process.

In [ ]:
print("Starting fine-tuning of GPT-2 on product reviews...")

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator
)

# Start training
trainer.train()

# Save the model locally
trainer.save_model("./gpt2-finetuned")
print("Fine-tuning completed and model saved!")

#Cell 4: Testing the Fine-Tuned Model
To verify that the model has learned the "language" of reviews, we provide it with a prompt related to electronics. We use sampling techniques like top_p and temperature to ensure the generated text is creative and doesn't get stuck in repetitive loops.

In [ ]:
print("\n=== Generated Product Reviews ===")

prompt = "This wireless earbuds are"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate text using the fine-tuned model
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Prompt: {prompt}")
print(f"Generated Result:\n{generated}")